In [ ]:
# ============================================================
# Self-contained DR_CF sensitivity diagnostic
# Generated from 01_simulation_study.ipynb + DR_CF diagnostic add-on
#
# This file does NOT run the original main simulation.
# It only runs the DR_CF sensitivity experiment for
#   e-truncation and bootstrap percentile intervals.
# ============================================================

# ============================================================
# ZIB + partial revelation:
#   STD_ZIB vs PR_MLE vs DR_CF
#   with cross-fitting and misspecified informative-H scenario
# One-cell Colab code
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit
from scipy.optimize import brentq, minimize
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# configuration
# ------------------------------------------------------------
OUTDIR = "zib_partial_revelation_cf_runs"
os.makedirs(OUTDIR, exist_ok=True)

# quick check:
# N=2500, MC_REPS=30
# paper-quality:
# N=4000, MC_REPS=100
N = 2500
MC_REPS = 10
RHO_GRID = [0.00, 0.01, 0.02, 0.05, 0.10, 0.20]

# structural truth for S|X and Y|S,X
# X = (1, x1, x2, x3)
ALPHA_TRUE = np.array([-0.8,  1.2, -0.9,  0.6], dtype=float)
BETA_TRUE  = np.array([ 0.2,  0.5,  1.0, -0.6], dtype=float)

BASE_SEED = 2026

# cross-fitting
N_SPLITS = 5

# DR proxy stabilization
CLIP_SE = True
SE_CLIP_LO = 1e-4
SE_CLIP_HI = 1 - 1e-4

# approximate interval for DR is sandwich-based
# for publication-level interval calibration, bootstrap is safer

SCENARIOS = ["clean", "informative_H"]

# optimizer guard against non-identified ridge runaway
# This is not a statistical constraint; it prevents failed BFGS starts
# from dominating Monte Carlo averages.
MAX_ABS_THETA = 30.0

# ------------------------------------------------------------
# helpers
# ------------------------------------------------------------
def add_intercept(X):
    return np.column_stack([np.ones(len(X)), X])

def pinv_psd(H, ridge=1e-8):
    Hs = 0.5 * (H + H.T)
    vals, vecs = np.linalg.eigh(Hs)
    vals2 = np.where(vals > ridge, vals, ridge)
    return (vecs / vals2) @ vecs.T

def numerical_hessian_from_grad(grad_fun, theta, eps=1e-4):
    d = len(theta)
    H = np.zeros((d, d))
    for j in range(d):
        e = np.zeros(d)
        e[j] = eps
        gp = grad_fun(theta + e)
        gm = grad_fun(theta - e)
        H[:, j] = (gp - gm) / (2 * eps)
    return 0.5 * (H + H.T)

def fit_bfgs(loss_grad_fun, theta0, maxiter=400):
    return minimize(
        fun=lambda th: loss_grad_fun(th)[0],
        x0=theta0,
        jac=lambda th: loss_grad_fun(th)[1],
        method="BFGS",
        options={"gtol": 1e-6, "maxiter": maxiter, "disp": False},
    )

def multi_start_fit(fun, starts, max_abs_theta=MAX_ABS_THETA, require_success=False):
    """
    Multi-start optimizer with a guard against non-identified-ridge runaway.

    The standard ZIB likelihood identifies only q(x)=pi(x)mu(x), not the
    decomposition into pi and mu.  Along nearly flat directions, BFGS can
    sometimes return an enormous parameter vector with a finite objective and
    success=False.  Such a run should not be allowed to dominate RMSE averages.
    """
    best = None
    best_any = None

    for th0 in starts:
        try:
            res = fit_bfgs(fun, th0)

            finite = np.isfinite(res.fun) and np.all(np.isfinite(res.x))
            bounded = finite and (np.max(np.abs(res.x)) <= max_abs_theta)
            successful = (bool(res.success) or not require_success)

            if finite:
                if best_any is None or res.fun < best_any.fun:
                    best_any = res

            if bounded and successful:
                if best is None or res.fun < best.fun:
                    best = res

        except Exception:
            pass

    if best is not None:
        return best

    # Fail fast instead of silently accepting an exploding ridge solution.
    if best_any is not None:
        raise RuntimeError(
            "Optimization produced only failed or exploding solutions; "
            f"best max|theta|={np.max(np.abs(best_any.x)):.3g}, "
            f"success={best_any.success}, message={best_any.message}"
        )

    raise RuntimeError("All optimization starts failed.")

def make_starts(p, seed=123):
    rng = np.random.default_rng(seed)
    z = np.zeros(2 * p)
    r1 = rng.normal(scale=0.7, size=2 * p)
    r2 = rng.normal(scale=1.2, size=2 * p)
    return [z, r1, r2, np.r_[r1[p:], r1[:p]], np.r_[r2[p:], r2[:p]]]

def calibrate_intercept(lp, target, lo=-50.0, hi=50.0):
    if target <= 0:
        return -50.0
    if target >= 1:
        return 50.0
    f = lambda c: expit(c + lp).mean() - target
    flo, fhi = f(lo), f(hi)
    if flo > 0:
        return lo
    if fhi < 0:
        return hi
    return brentq(f, lo, hi)

# ------------------------------------------------------------
# data generation
# ------------------------------------------------------------
def simulate_structural_zib(n, alpha, beta, seed):
    """
    Structural model:
      P(S=1|X)=logit^{-1}(X alpha)
      P(Y=1|S=1,X)=logit^{-1}(X beta)

    This function is intentionally scenario-free.  The two scenarios must share
    the same (X,S,Y) in each Monte Carlo replication; only H and the revelation
    mechanism differ.
    """
    rng = np.random.default_rng(seed)

    xraw = rng.normal(size=(n, 3))
    X = add_intercept(xraw)

    pi = expit(X @ alpha)
    mu = expit(X @ beta)

    S = rng.binomial(1, pi)
    Y = rng.binomial(1, S * mu)

    return {
        "X": X,
        "xraw": xraw,
        "S": S,
        "Y": Y,
        "pi": pi,
        "mu": mu,
        "alpha_true": alpha.copy(),
        "beta_true": beta.copy(),
    }

def add_auxiliary_history(structural, scenario, seed):
    """
    Add scenario-specific auxiliary history H without changing (X,S,Y).
    """
    rng = np.random.default_rng(seed)

    base = dict(structural)
    xraw = structural["xraw"]
    S = structural["S"]

    if scenario == "clean":
        H = xraw.copy()

    elif scenario == "informative_H":
        b = 1.5 * S + 0.6 * xraw[:, 0] - 0.4 * xraw[:, 1] + rng.normal(scale=0.7, size=len(S))
        H = np.column_stack([xraw, b])

    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    base.update({
        "H": H,
        "scenario": scenario,
    })
    return base

def simulate_base_zib(n, alpha, beta, scenario, seed):
    """
    Backward-compatible wrapper.  Prefer simulate_structural_zib +
    add_auxiliary_history inside the Monte Carlo loop.
    """
    structural = simulate_structural_zib(n, alpha, beta, seed)
    return add_auxiliary_history(structural, scenario, seed + 7919)

def attach_partial_revelation(base, rho_target, seed):
    """
    Generate revelation indicator R among zeros only.
    clean:
      R depends on X only.
    informative_H:
      R depends strongly on extra informative history H[:,3],
      inducing selection among zero outcomes.

    This function does not change X, S, or Y.
    """
    rng = np.random.default_rng(seed)

    H = base["H"]
    Y = base["Y"]
    S = base["S"]
    scenario = base["scenario"]

    zero = (Y == 0)

    if rho_target <= 0:
        R = np.zeros(len(Y), dtype=int)
        e0 = np.zeros(len(Y), dtype=float)
        W = R * S
        out = dict(base)
        out.update({
            "R": R,
            "W": W,
            "e0": e0,
            "rho_realized": 0.0,
        })
        return out

    if scenario == "clean":
        lp = 0.9 * H[:, 0] - 0.7 * H[:, 1] + 0.4 * H[:, 2]

    elif scenario == "informative_H":
        lp = 0.5 * H[:, 0] - 0.3 * H[:, 1] + 1.8 * H[:, 3]

    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    c = calibrate_intercept(lp[zero], rho_target) if zero.sum() > 0 else -50.0
    e0 = expit(c + lp)

    R = np.zeros(len(Y), dtype=int)
    if zero.sum() > 0:
        R[zero] = rng.binomial(1, e0[zero])

    W = R * S

    out = dict(base)
    out.update({
        "R": R,
        "W": W,
        "e0": e0,
        "rho_realized": float(R[zero].mean()) if zero.sum() > 0 else 0.0,
    })
    return out

# ------------------------------------------------------------
# losses / gradients
# ------------------------------------------------------------
def standard_loss_grad(theta, X, Y):
    """
    Standard ZIB:
      sum_i [ y log(pi*mu) + (1-y) log(1-pi*mu) ]
    """
    p = X.shape[1]
    alpha = theta[:p]
    beta = theta[p:]

    pi = expit(X @ alpha)
    mu = expit(X @ beta)
    q = np.clip(pi * mu, 1e-12, 1 - 1e-12)

    ll = np.sum(Y * np.log(q) + (1 - Y) * np.log(1 - q))

    wa = (Y - q) * (1 - pi) / np.clip(1 - q, 1e-12, None)
    wb = (Y - q) * (1 - mu) / np.clip(1 - q, 1e-12, None)

    grad = np.concatenate([X.T @ wa, X.T @ wb])
    return -ll, -grad

def pr_loss_grad(theta, X, Y, R, W):
    """
    Naive partial-revelation likelihood ignoring H:
      Y=1                    -> log pi + log mu
      Y=0, R=1, W=1          -> log pi + log(1-mu)
      Y=0, R=1, W=0          -> log(1-pi)
      Y=0, R=0               -> log(1-pi*mu)
    In the clean scenario this is well-aligned.
    In informative_H it becomes misspecified if H is ignored.
    """
    p = X.shape[1]
    alpha = theta[:p]
    beta = theta[p:]

    pi = expit(X @ alpha)
    mu = expit(X @ beta)
    q = np.clip(pi * mu, 1e-12, 1 - 1e-12)

    y1 = (Y == 1)
    z0 = (Y == 0)
    zr1 = z0 & (R == 1) & (W == 1)
    zr0 = z0 & (R == 1) & (W == 0)
    zu  = z0 & (R == 0)

    ll = 0.0
    ga = np.zeros(p)
    gb = np.zeros(p)

    if np.any(y1):
        ll += np.sum(np.log(np.clip(pi[y1], 1e-12, None)) + np.log(np.clip(mu[y1], 1e-12, None)))
        ga += X[y1].T @ (1 - pi[y1])
        gb += X[y1].T @ (1 - mu[y1])

    if np.any(zr1):
        ll += np.sum(np.log(np.clip(pi[zr1], 1e-12, None)) + np.log(np.clip(1 - mu[zr1], 1e-12, None)))
        ga += X[zr1].T @ (1 - pi[zr1])
        gb += X[zr1].T @ (-mu[zr1])

    if np.any(zr0):
        ll += np.sum(np.log(np.clip(1 - pi[zr0], 1e-12, None)))
        ga += X[zr0].T @ (-pi[zr0])

    if np.any(zu):
        ll += np.sum(np.log(np.clip(1 - q[zu], 1e-12, None)))
        ga += X[zu].T @ (-(q[zu] * (1 - pi[zu]) / np.clip(1 - q[zu], 1e-12, None)))
        gb += X[zu].T @ (-(q[zu] * (1 - mu[zu]) / np.clip(1 - q[zu], 1e-12, None)))

    grad = np.concatenate([ga, gb])
    return -ll, -grad

# ------------------------------------------------------------
# nuisance fitting with cross-fitting
# ------------------------------------------------------------
def fit_prob_model(X_train, y_train, X_pred, seed=0):
    """
    Flexible binary probability model with fallback.
    """
    y_train = np.asarray(y_train).astype(int)

    if len(y_train) == 0:
        return np.full(len(X_pred), 0.5)

    u = np.unique(y_train)
    if len(u) < 2:
        p = np.clip(y_train.mean(), 1e-4, 1 - 1e-4)
        return np.full(len(X_pred), p)

    # flexible first choice
    try:
        clf = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=3,
            max_iter=250,
            min_samples_leaf=20,
            random_state=seed
        )
        clf.fit(X_train, y_train)
        p = clf.predict_proba(X_pred)[:, 1]
        return np.clip(p, 1e-4, 1 - 1e-4)
    except Exception:
        pass

    # fallback
    clf = LogisticRegression(C=2.0, solver="lbfgs", max_iter=2000)
    clf.fit(X_train, y_train)
    p = clf.predict_proba(X_pred)[:, 1]
    return np.clip(p, 1e-4, 1 - 1e-4)

def crossfit_em(data, n_splits=5, seed=0):
    """
    Cross-fitted estimates of
      e(H) = P(R=1 | Y=0, H)
      m(H) = E(S | Y=0, H)
    where m is trained only on revealed zeros.
    """
    H = data["H"]
    Y = data["Y"]
    R = data["R"]
    W = data["W"]

    n = len(Y)
    ehat = np.ones(n)
    mhat = np.ones(n)

    zero_idx = np.where(Y == 0)[0]
    if len(zero_idx) == 0:
        return ehat, mhat

    n_splits_eff = min(n_splits, len(zero_idx))
    if n_splits_eff < 2:
        # fallback to no split
        ehat[zero_idx] = fit_prob_model(H[zero_idx], R[zero_idx], H[zero_idx], seed=seed)
        rev = zero_idx[R[zero_idx] == 1]
        mhat[zero_idx] = fit_prob_model(H[rev], W[rev], H[zero_idx], seed=seed+1) if len(rev) > 0 else 0.5
        return np.clip(ehat, 1e-3, 1 - 1e-3), np.clip(mhat, 1e-4, 1 - 1e-4)

    kf = KFold(n_splits=n_splits_eff, shuffle=True, random_state=seed)

    for fold_id, (tr_pos, te_pos) in enumerate(kf.split(zero_idx)):
        tr = zero_idx[tr_pos]
        te = zero_idx[te_pos]

        # e(H): use all zero outcomes in train fold
        ehat[te] = fit_prob_model(H[tr], R[tr], H[te], seed=seed + 100 * fold_id + 1)

        # m(H): use only revealed zeros in train fold
        rev_tr = tr[R[tr] == 1]
        if len(rev_tr) == 0:
            mhat[te] = 0.5
        else:
            mhat[te] = fit_prob_model(H[rev_tr], W[rev_tr], H[te], seed=seed + 100 * fold_id + 2)

    return np.clip(ehat, 1e-3, 1 - 1e-3), np.clip(mhat, 1e-4, 1 - 1e-4)

def build_Se(Y, R, W, ehat, mhat, clip=True):
    Se = Y + (1 - Y) * (R * (W - mhat) / np.clip(ehat, 1e-12, None) + mhat)
    if clip:
        Se = np.clip(Se, SE_CLIP_LO, SE_CLIP_HI)
    return Se

def dr_pseudo_loss_grad(theta, X, Y, Se):
    """
    DR pseudo-likelihood special case:
      l_dr = sum [ Se log pi + (1-Se) log(1-pi) ]
           + sum [ Y log mu + (Se-Y) log(1-mu) ]
    """
    p = X.shape[1]
    alpha = theta[:p]
    beta = theta[p:]

    pi = expit(X @ alpha)
    mu = expit(X @ beta)
    Se = np.clip(Se, 1e-12, 1 - 1e-12)

    ll = np.sum(Se * np.log(np.clip(pi, 1e-12, None)) + (1 - Se) * np.log(np.clip(1 - pi, 1e-12, None)))
    ll += np.sum(Y * np.log(np.clip(mu, 1e-12, None)) + (Se - Y) * np.log(np.clip(1 - mu, 1e-12, None)))

    grad = np.concatenate([X.T @ (Se - pi), X.T @ (Y - Se * mu)])
    return -ll, -grad

# ------------------------------------------------------------
# fitters
# ------------------------------------------------------------
def fit_standard_once(base, start_seed=123):
    X = base["X"]
    Y = base["Y"]
    p = X.shape[1]

    starts = make_starts(p, seed=start_seed)
    res = multi_start_fit(lambda th: standard_loss_grad(th, X, Y), starts)
    theta_hat = res.x

    H = numerical_hessian_from_grad(lambda th: standard_loss_grad(th, X, Y)[1], theta_hat)
    cov = pinv_psd(H)
    se = np.sqrt(np.diag(cov))

    return {"theta": theta_hat, "H": H, "se": se}

def fit_pr_mle(data, theta_init):
    X = data["X"]
    Y = data["Y"]
    R = data["R"]
    W = data["W"]
    p = X.shape[1]

    starts = [theta_init, np.zeros(2 * p), np.r_[theta_init[p:], theta_init[:p]]]
    res = multi_start_fit(lambda th: pr_loss_grad(th, X, Y, R, W), starts)
    theta = res.x

    H = numerical_hessian_from_grad(lambda th: pr_loss_grad(th, X, Y, R, W)[1], theta)
    cov = pinv_psd(H)
    se = np.sqrt(np.diag(cov))

    return {"theta": theta, "H": H, "se": se}

def fit_dr_cf(data, theta_init, n_splits=5, seed=0):
    """
    DR with cross-fitted nuisances.
    By design, skip rho=0 in main comparison.
    """
    X = data["X"]
    Y = data["Y"]
    R = data["R"]
    W = data["W"]
    p = X.shape[1]

    if R.sum() == 0:
        return None

    ehat, mhat = crossfit_em(data, n_splits=n_splits, seed=seed)
    Se = build_Se(Y, R, W, ehat, mhat, clip=CLIP_SE)

    starts = [theta_init, np.zeros(2 * p)]
    res = multi_start_fit(lambda th: dr_pseudo_loss_grad(th, X, Y, Se), starts)
    theta = res.x

    alpha_hat = theta[:p]
    beta_hat = theta[p:]
    pi_hat = expit(X @ alpha_hat)
    mu_hat = expit(X @ beta_hat)

    # block-diagonal sensitivity
    A = np.block([
        [X.T @ ((pi_hat * (1 - pi_hat))[:, None] * X), np.zeros((p, p))],
        [np.zeros((p, p)), X.T @ ((np.clip(Se, 1e-12, None) * mu_hat * (1 - mu_hat))[:, None] * X)]
    ])

    # empirical score outer product
    score_i = np.concatenate([
        ((Se - pi_hat)[:, None] * X),
        ((Y - Se * mu_hat)[:, None] * X)
    ], axis=1)

    B = score_i.T @ score_i
    cov = pinv_psd(A) @ B @ pinv_psd(A)
    se = np.sqrt(np.diag(np.abs(cov)))

    return {
        "theta": theta,
        "H": A,      # sensitivity, not exact likelihood Hessian
        "se": se,
        "Se_min": float(Se.min()),
        "Se_max": float(Se.max()),
        "ehat_mean_zero": float(ehat[Y == 0].mean()),
        "mhat_mean_zero": float(mhat[Y == 0].mean()),
    }

# ------------------------------------------------------------
# metrics
# ------------------------------------------------------------
def record_metrics(scenario, method, fit, rho, rep, theta_true, p):
    theta_hat = fit["theta"]
    se = np.clip(fit["se"], 1e-8, np.inf)
    err = theta_hat - theta_true

    d_swap = np.r_[np.ones(p), -np.ones(p)] / np.sqrt(2 * p)

    out = {
        "scenario": scenario,
        "rho": float(rho),
        "rep": int(rep),
        "method": method,
        "rmse_all": float(np.sqrt(np.mean(err**2))),
        "mean_abs_bias": float(np.mean(np.abs(err))),
        "coverage_mean": float(((theta_true >= theta_hat - 1.96 * se) &
                                (theta_true <= theta_hat + 1.96 * se)).mean()),
        "min_eig": float(np.linalg.eigvalsh(0.5 * (fit["H"] + fit["H"].T)).min()),
        "swap_proj": float(d_swap @ theta_hat),
        "alpha1_hat": float(theta_hat[1]),
        "beta1_hat": float(theta_hat[p + 1]),
    }
    return out

# ------------------------------------------------------------
# plotting
# ------------------------------------------------------------
def summarize_results(df):
    return (
        df.groupby(["scenario", "rho", "method"], as_index=False)
          .agg(
              rmse_mean=("rmse_all", "mean"),
              rmse_sd=("rmse_all", "std"),
              bias_mean=("mean_abs_bias", "mean"),
              bias_sd=("mean_abs_bias", "std"),
              cover_mean=("coverage_mean", "mean"),
              cover_sd=("coverage_mean", "std"),
              min_eig_mean=("min_eig", "mean"),
              min_eig_sd=("min_eig", "std"),
              swap_var=("swap_proj", "var"),
              alpha1_mean=("alpha1_hat", "mean"),
              beta1_mean=("beta1_hat", "mean"),
          )
    )

def plot_curve(summary, scenario, ycol, ylabel, filename, methods):
    plt.figure(figsize=(7, 5))
    sub_all = summary[summary["scenario"] == scenario]
    for method in methods:
        sub = sub_all[sub_all["method"] == method].sort_values("rho")
        if len(sub) == 0:
            continue
        plt.plot(sub["rho"], sub[ycol], marker="o", label=method)
    plt.xlabel("P(R=1)")
    plt.ylabel(ylabel)
    plt.title(scenario)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=180)
    plt.show()

def plot_scatter(df, scenario, rho_values=(0.00, 0.05), methods=("STD_ZIB", "PR_MLE", "DR_CF"), filename="scatter.png"):
    fig, axes = plt.subplots(len(rho_values), len(methods), figsize=(4.3 * len(methods), 4.0 * len(rho_values)))

    if len(rho_values) == 1 and len(methods) == 1:
        axes = np.array([[axes]])
    elif len(rho_values) == 1:
        axes = np.array([axes])
    elif len(methods) == 1:
        axes = np.array([[ax] for ax in axes])

    for i, rho in enumerate(rho_values):
        for j, method in enumerate(methods):
            ax = axes[i, j]
            sub = df[(df["scenario"] == scenario) & (df["rho"] == rho) & (df["method"] == method)]
            if len(sub) > 0:
                ax.scatter(sub["alpha1_hat"], sub["beta1_hat"], alpha=0.7)
            ax.axvline(ALPHA_TRUE[1], linestyle="--")
            ax.axhline(BETA_TRUE[1], linestyle="--")
            ax.set_title(f"{method}, rho={rho:.2f}")
            ax.set_xlabel("alpha1 hat")
            ax.set_ylabel("beta1 hat")
            ax.grid(alpha=0.3)

    plt.suptitle(scenario, y=1.02)
    plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches="tight")
    plt.show()

# ------------------------------------------------------------
# main simulation
# ------------------------------------------------------------
def run_simulation():
    theta_true = np.r_[ALPHA_TRUE, BETA_TRUE]
    p = len(ALPHA_TRUE)

    rows = []

    for rep in range(MC_REPS):
        if (rep + 1) % 5 == 0 or rep == 0:
            print(f"rep {rep+1}/{MC_REPS}")

        # Generate the structural data once.  Both scenarios use this same
        # (X,S,Y), so STD_ZIB is a common baseline by construction.
        structural = simulate_structural_zib(
            n=N,
            alpha=ALPHA_TRUE,
            beta=BETA_TRUE,
            seed=BASE_SEED + 1000 * rep
        )

        std_fit = fit_standard_once(structural, start_seed=BASE_SEED + rep)

        for scenario in SCENARIOS:
            base = add_auxiliary_history(
                structural,
                scenario=scenario,
                seed=BASE_SEED + 1000 * rep + 10000 * SCENARIOS.index(scenario) + 17
            )

            for rho in RHO_GRID:
                rows.append(record_metrics(scenario, "STD_ZIB", std_fit, rho, rep, theta_true, p))

            for rho in RHO_GRID:
                data = attach_partial_revelation(
                    base,
                    rho_target=rho,
                    seed=BASE_SEED + 1000 * rep + 10000 * SCENARIOS.index(scenario) + int(1e5 * rho) + 13
                )

                # PR_MLE
                if abs(rho) < 1e-15:
                    pr_fit = std_fit
                else:
                    pr_fit = fit_pr_mle(data, theta_init=std_fit["theta"])
                rows.append(record_metrics(scenario, "PR_MLE", pr_fit, rho, rep, theta_true, p))

                # DR_CF: skip rho=0 by design
                if rho > 0:
                    dr_fit = fit_dr_cf(
                        data,
                        theta_init=pr_fit["theta"],
                        n_splits=N_SPLITS,
                        seed=BASE_SEED + 1000 * rep + 10000 * SCENARIOS.index(scenario) + int(1e5 * rho) + 97
                    )
                    if dr_fit is not None:
                        rows.append(record_metrics(scenario, "DR_CF", dr_fit, rho, rep, theta_true, p))

    return pd.DataFrame(rows)


# ============================================================
# DR_CF finite-sample diagnostic add-on:
#   propensity truncation sensitivity + nonparametric bootstrap percentile CI
#
# How to use in Colab:
#   1. In 01_simulation_study.ipynb, run the definitions above the final
#      "df = run_simulation()" block, or run the original first cell once.
#   2. Paste and run this cell.
#
# Recommended pilot:
#   MC_REPS_SENS = 30 or 50, BOOT_REPS = 100
# Paper-quality diagnostic:
#   MC_REPS_SENS = 100, BOOT_REPS = 200 or 399
# ============================================================

import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.special import expit
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from joblib import Parallel, delayed

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ------------------------------------------------------------
# diagnostic configuration
# ------------------------------------------------------------
OUTDIR_SENS = "zib_partial_revelation_dr_cf_sensitivity"
os.makedirs(OUTDIR_SENS, exist_ok=True)

# Keep this moderate for the first run.  Increase after checking runtime.
MC_REPS_SENS = 50
BOOT_REPS = 30

# Colab CPU workers
N_JOBS = 7

# Use the same N as the main notebook unless overridden.
try:
    N_SENS = N
except NameError:
    N_SENS = 2500

# Structural truth.  These defaults match 01_simulation_study.ipynb.
# They are used if this diagnostic cell is run without first executing
# the original simulation-definition cell.
try:
    ALPHA_TRUE
except NameError:
    ALPHA_TRUE = np.array([-0.8,  1.2, -0.9,  0.6], dtype=float)

try:
    BETA_TRUE
except NameError:
    BETA_TRUE = np.array([ 0.2,  0.5,  1.0, -0.6], dtype=float)

# DR_CF is not used at rho=0.
RHO_GRID_SENS = [0.01, 0.02, 0.05, 0.10, 0.20]
SCENARIOS_SENS = ["clean", "informative_H"]

# Propensity truncation levels for e(H)=P(R=1|Y=0,H).
# 0.001 matches the current loose setting; 0.01 and 0.05 are aggressive.
E_TRUNC_GRID = [0.001, 0.01, 0.05]

# Nuisance learners to compare.
# "hgb" is close to the current flexible default.
# "logistic" is a simpler parametric benchmark.
# "rf" is optional and slower; include only if desired.
NUISANCE_KINDS = ["hgb", "logistic"]
# NUISANCE_KINDS = ["hgb", "logistic", "rf"]

N_SPLITS_SENS = 5
BASE_SEED_SENS = 2026

# Percentile CI level
CI_LEVEL = 0.95
CI_ALPHA = 1.0 - CI_LEVEL

# Keep Se clipping fixed, so the sensitivity is primarily about e-truncation.
SE_CLIP_LO_SENS = 1e-4
SE_CLIP_HI_SENS = 1.0 - 1e-4
M_CLIP_SENS = 1e-4


# ------------------------------------------------------------
# small utilities
# ------------------------------------------------------------
def _subset_data_dict(data, idx):
    """Bootstrap/subset all observation-level arrays in the data dictionary."""
    idx = np.asarray(idx)
    out = {}
    n = len(data["Y"])
    for k, v in data.items():
        if isinstance(v, np.ndarray) and len(v) == n:
            out[k] = v[idx]
        else:
            out[k] = v
    return out


def _safe_mean_coverage(theta_true, lo, hi):
    ok = np.isfinite(lo) & np.isfinite(hi)
    if ok.sum() == 0:
        return np.nan
    return float(((theta_true[ok] >= lo[ok]) & (theta_true[ok] <= hi[ok])).mean())


def _ci_width(lo, hi):
    ok = np.isfinite(lo) & np.isfinite(hi)
    if ok.sum() == 0:
        return np.nan
    return float(np.nanmean(hi[ok] - lo[ok]))


# ------------------------------------------------------------
# nuisance models with configurable learner and truncation
# ------------------------------------------------------------
def fit_prob_model_sens(X_train, y_train, X_pred, seed=0, kind="hgb", clip=1e-4):
    """
    Estimate binary probabilities.  The output is clipped only at `clip`.

    kind:
      - "logistic": parametric logistic regression
      - "hgb": histogram gradient boosting
      - "rf": random forest
    """
    y_train = np.asarray(y_train).astype(int)

    if len(y_train) == 0:
        return np.full(len(X_pred), 0.5)

    u = np.unique(y_train)
    if len(u) < 2:
        p0 = np.clip(y_train.mean(), clip, 1.0 - clip)
        return np.full(len(X_pred), p0)

    try:
        if kind == "logistic":
            clf = LogisticRegression(
                C=2.0,
                solver="lbfgs",
                max_iter=2000
            )

        elif kind == "hgb":
            clf = HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_depth=3,
                max_iter=250,
                min_samples_leaf=20,
                l2_regularization=0.0,
                random_state=seed
            )

        elif kind == "rf":
            clf = RandomForestClassifier(
                n_estimators=250,
                max_depth=None,
                min_samples_leaf=10,
                max_features="sqrt",
                n_jobs=1,
                random_state=seed
            )

        else:
            raise ValueError(f"Unknown nuisance kind: {kind}")

        clf.fit(X_train, y_train)
        p = clf.predict_proba(X_pred)[:, 1]
        return np.clip(p, clip, 1.0 - clip)

    except Exception:
        # robust fallback
        clf = LogisticRegression(C=2.0, solver="lbfgs", max_iter=2000)
        clf.fit(X_train, y_train)
        p = clf.predict_proba(X_pred)[:, 1]
        return np.clip(p, clip, 1.0 - clip)


def crossfit_em_sens(data, n_splits=5, seed=0, e_trunc=0.001, nuisance_kind="hgb"):
    """
    Cross-fitted estimates of
      e(H) = P(R=1 | Y=0,H),
      m(H) = E(S | Y=0,H),
    with configurable e-truncation and nuisance learner.
    """
    H = data["H"]
    Y = data["Y"]
    R = data["R"]
    W = data["W"]

    n = len(Y)
    ehat = np.ones(n)
    mhat = np.ones(n)

    zero_idx = np.where(Y == 0)[0]
    if len(zero_idx) == 0:
        return ehat, mhat

    n_splits_eff = min(n_splits, len(zero_idx))
    if n_splits_eff < 2:
        ehat[zero_idx] = fit_prob_model_sens(
            H[zero_idx], R[zero_idx], H[zero_idx],
            seed=seed, kind=nuisance_kind, clip=e_trunc
        )
        rev = zero_idx[R[zero_idx] == 1]
        if len(rev) > 0:
            mhat[zero_idx] = fit_prob_model_sens(
                H[rev], W[rev], H[zero_idx],
                seed=seed + 1, kind=nuisance_kind, clip=M_CLIP_SENS
            )
        else:
            mhat[zero_idx] = 0.5

        return (
            np.clip(ehat, e_trunc, 1.0 - e_trunc),
            np.clip(mhat, M_CLIP_SENS, 1.0 - M_CLIP_SENS)
        )

    kf = KFold(n_splits=n_splits_eff, shuffle=True, random_state=seed)

    for fold_id, (tr_pos, te_pos) in enumerate(kf.split(zero_idx)):
        tr = zero_idx[tr_pos]
        te = zero_idx[te_pos]

        ehat[te] = fit_prob_model_sens(
            H[tr], R[tr], H[te],
            seed=seed + 100 * fold_id + 1,
            kind=nuisance_kind,
            clip=e_trunc
        )

        rev_tr = tr[R[tr] == 1]
        if len(rev_tr) == 0:
            mhat[te] = 0.5
        else:
            mhat[te] = fit_prob_model_sens(
                H[rev_tr], W[rev_tr], H[te],
                seed=seed + 100 * fold_id + 2,
                kind=nuisance_kind,
                clip=M_CLIP_SENS
            )

    return (
        np.clip(ehat, e_trunc, 1.0 - e_trunc),
        np.clip(mhat, M_CLIP_SENS, 1.0 - M_CLIP_SENS)
    )


def build_Se_sens(Y, R, W, ehat, mhat):
    """
    DR pseudo-label for S.
    Only ehat is varied by e_trunc; Se clipping is held fixed.
    """
    Se = Y + (1 - Y) * (R * (W - mhat) / np.clip(ehat, 1e-12, None) + mhat)
    return np.clip(Se, SE_CLIP_LO_SENS, SE_CLIP_HI_SENS)


# ------------------------------------------------------------
# DR_CF fit with configurable truncation and nuisance learner
# ------------------------------------------------------------
def fit_dr_cf_sens(
    data,
    theta_init,
    n_splits=5,
    seed=0,
    e_trunc=0.001,
    nuisance_kind="hgb"
):
    """
    Same target estimating equation as the original DR_CF, but with
    configurable e-truncation and nuisance learner.
    """
    X = data["X"]
    Y = data["Y"]
    R = data["R"]
    p = X.shape[1]

    if R.sum() == 0:
        return None

    ehat, mhat = crossfit_em_sens(
        data,
        n_splits=n_splits,
        seed=seed,
        e_trunc=e_trunc,
        nuisance_kind=nuisance_kind
    )
    Se = build_Se_sens(Y, R, data["W"], ehat, mhat)

    starts = [theta_init, np.zeros(2 * p)]
    res = multi_start_fit(lambda th: dr_pseudo_loss_grad(th, X, Y, Se), starts)
    theta = res.x

    alpha_hat = theta[:p]
    beta_hat = theta[p:]

    pi_hat = expit(X @ alpha_hat)
    mu_hat = expit(X @ beta_hat)

    A = np.block([
        [X.T @ ((pi_hat * (1.0 - pi_hat))[:, None] * X), np.zeros((p, p))],
        [np.zeros((p, p)),
         X.T @ ((np.clip(Se, 1e-12, None) * mu_hat * (1.0 - mu_hat))[:, None] * X)]
    ])

    score_i = np.concatenate([
        ((Se - pi_hat)[:, None] * X),
        ((Y - Se * mu_hat)[:, None] * X)
    ], axis=1)

    B = score_i.T @ score_i
    cov = pinv_psd(A) @ B @ pinv_psd(A)
    se = np.sqrt(np.diag(np.abs(cov)))

    # ESS for the revelation-IPW component among observed zeros.
    zero = (Y == 0)
    rz = zero & (R == 1)
    weights = np.zeros(len(Y))
    weights[rz] = 1.0 / np.clip(ehat[rz], 1e-12, None)
    ess = (weights.sum() ** 2) / np.sum(weights ** 2) if np.sum(weights ** 2) > 0 else 0.0

    return {
        "theta": theta,
        "H": A,
        "se": se,
        "ehat_min_zero": float(np.min(ehat[zero])) if zero.sum() > 0 else np.nan,
        "ehat_q01_zero": float(np.quantile(ehat[zero], 0.01)) if zero.sum() > 0 else np.nan,
        "ehat_q05_zero": float(np.quantile(ehat[zero], 0.05)) if zero.sum() > 0 else np.nan,
        "ehat_mean_zero": float(np.mean(ehat[zero])) if zero.sum() > 0 else np.nan,
        "mhat_mean_zero": float(np.mean(mhat[zero])) if zero.sum() > 0 else np.nan,
        "Se_min": float(np.min(Se)),
        "Se_max": float(np.max(Se)),
        "ess_revealed": float(ess),
        "n_revealed": int(R.sum()),
        "rho_realized": float(data.get("rho_realized", np.nan)),
    }


def bootstrap_percentile_ci_dr_cf(
    data,
    theta_init,
    B=100,
    seed=0,
    e_trunc=0.001,
    nuisance_kind="hgb",
    n_splits=5
):
    """
    Pairs bootstrap percentile interval for DR_CF.
    The nuisances are re-estimated inside every bootstrap sample.
    """
    rng = np.random.default_rng(seed)
    n = len(data["Y"])
    theta_boot = []
    fail = 0

    for b in range(B):
        idx = rng.integers(0, n, size=n)
        db = _subset_data_dict(data, idx)

        try:
            fit_b = fit_dr_cf_sens(
                db,
                theta_init=theta_init,
                n_splits=n_splits,
                seed=seed + 1000 * b + 77,
                e_trunc=e_trunc,
                nuisance_kind=nuisance_kind
            )
            if fit_b is None:
                fail += 1
                continue
            th = fit_b["theta"]
            if np.all(np.isfinite(th)):
                theta_boot.append(th)
            else:
                fail += 1
        except Exception:
            fail += 1

    if len(theta_boot) < max(20, int(0.5 * B)):
        p = len(theta_init)
        return {
            "lo": np.full(p, np.nan),
            "hi": np.full(p, np.nan),
            "sd": np.full(p, np.nan),
            "n_success": len(theta_boot),
            "n_fail": fail,
        }

    theta_boot = np.asarray(theta_boot)
    lo = np.quantile(theta_boot, CI_ALPHA / 2.0, axis=0)
    hi = np.quantile(theta_boot, 1.0 - CI_ALPHA / 2.0, axis=0)
    sd = np.std(theta_boot, axis=0, ddof=1)

    return {
        "lo": lo,
        "hi": hi,
        "sd": sd,
        "n_success": int(len(theta_boot)),
        "n_fail": int(fail),
    }


# ------------------------------------------------------------
# one Monte Carlo task
# ------------------------------------------------------------
def run_one_dr_cf_sensitivity_task(rep, scenario, rho, nuisance_kind):
    """
    One task returns rows over E_TRUNC_GRID for a fixed
    (rep, scenario, rho, nuisance_kind).
    """
    theta_true = np.r_[ALPHA_TRUE, BETA_TRUE]
    p = len(ALPHA_TRUE)

    # Same structural seed convention as the main notebook.
    structural = simulate_structural_zib(
        n=N_SENS,
        alpha=ALPHA_TRUE,
        beta=BETA_TRUE,
        seed=BASE_SEED_SENS + 1000 * rep
    )

    std_fit = fit_standard_once(structural, start_seed=BASE_SEED_SENS + rep)

    base = add_auxiliary_history(
        structural,
        scenario=scenario,
        seed=BASE_SEED_SENS + 1000 * rep + 10000 * SCENARIOS_SENS.index(scenario) + 17
    )

    data = attach_partial_revelation(
        base,
        rho_target=rho,
        seed=BASE_SEED_SENS + 1000 * rep + 10000 * SCENARIOS_SENS.index(scenario) + int(1e5 * rho) + 13
    )

    # Use PR_MLE only as a stable starting value, matching the original workflow.
    pr_fit = fit_pr_mle(data, theta_init=std_fit["theta"])

    rows = []

    for e_trunc in E_TRUNC_GRID:
        try:
            fit = fit_dr_cf_sens(
                data,
                theta_init=pr_fit["theta"],
                n_splits=N_SPLITS_SENS,
                seed=BASE_SEED_SENS + 1000 * rep + 10000 * SCENARIOS_SENS.index(scenario) + int(1e5 * rho) + 97,
                e_trunc=e_trunc,
                nuisance_kind=nuisance_kind
            )

            if fit is None:
                continue

            theta_hat = fit["theta"]
            se = np.clip(fit["se"], 1e-8, np.inf)

            sandwich_lo = theta_hat - 1.96 * se
            sandwich_hi = theta_hat + 1.96 * se

            boot = bootstrap_percentile_ci_dr_cf(
                data,
                theta_init=theta_hat,   # start bootstrap near the DR solution
                B=BOOT_REPS,
                seed=BASE_SEED_SENS + 900000 + 1000 * rep + int(1e5 * rho) + 37,
                e_trunc=e_trunc,
                nuisance_kind=nuisance_kind,
                n_splits=N_SPLITS_SENS
            )

            boot_lo = boot["lo"]
            boot_hi = boot["hi"]

            err = theta_hat - theta_true
            d_swap = np.r_[np.ones(p), -np.ones(p)] / np.sqrt(2 * p)

            rows.append({
                "scenario": scenario,
                "rho": float(rho),
                "rho_realized": fit["rho_realized"],
                "rep": int(rep),
                "method": "DR_CF",
                "nuisance": nuisance_kind,
                "e_trunc": float(e_trunc),

                "rmse_all": float(np.sqrt(np.mean(err ** 2))),
                "mean_abs_bias": float(np.mean(np.abs(err))),

                "coverage_sandwich": _safe_mean_coverage(theta_true, sandwich_lo, sandwich_hi),
                "coverage_boot_pct": _safe_mean_coverage(theta_true, boot_lo, boot_hi),

                "width_sandwich": _ci_width(sandwich_lo, sandwich_hi),
                "width_boot_pct": _ci_width(boot_lo, boot_hi),

                "boot_success": int(boot["n_success"]),
                "boot_fail": int(boot["n_fail"]),

                "min_eig": float(np.linalg.eigvalsh(0.5 * (fit["H"] + fit["H"].T)).min()),
                "swap_proj": float(d_swap @ theta_hat),
                "alpha1_hat": float(theta_hat[1]),
                "beta1_hat": float(theta_hat[p + 1]),

                "ehat_min_zero": fit["ehat_min_zero"],
                "ehat_q01_zero": fit["ehat_q01_zero"],
                "ehat_q05_zero": fit["ehat_q05_zero"],
                "ehat_mean_zero": fit["ehat_mean_zero"],
                "mhat_mean_zero": fit["mhat_mean_zero"],
                "Se_min": fit["Se_min"],
                "Se_max": fit["Se_max"],
                "ess_revealed": fit["ess_revealed"],
                "n_revealed": fit["n_revealed"],
            })

        except Exception as exc:
            rows.append({
                "scenario": scenario,
                "rho": float(rho),
                "rep": int(rep),
                "method": "DR_CF",
                "nuisance": nuisance_kind,
                "e_trunc": float(e_trunc),
                "error": str(exc)[:200],
            })

    return rows


def summarize_dr_cf_sensitivity(df_sens):
    ok = df_sens.dropna(subset=["rmse_all"]).copy()
    if ok.empty:
        return ok

    return (
        ok.groupby(["scenario", "rho", "nuisance", "e_trunc"], as_index=False)
          .agg(
              n_rep=("rep", "nunique"),
              rmse_mean=("rmse_all", "mean"),
              rmse_sd=("rmse_all", "std"),
              bias_mean=("mean_abs_bias", "mean"),
              cover_sandwich=("coverage_sandwich", "mean"),
              cover_boot_pct=("coverage_boot_pct", "mean"),
              width_sandwich=("width_sandwich", "mean"),
              width_boot_pct=("width_boot_pct", "mean"),
              boot_success_mean=("boot_success", "mean"),
              min_eig_mean=("min_eig", "mean"),
              ess_mean=("ess_revealed", "mean"),
              n_revealed_mean=("n_revealed", "mean"),
              ehat_q01_mean=("ehat_q01_zero", "mean"),
              ehat_q05_mean=("ehat_q05_zero", "mean"),
          )
    )


def run_dr_cf_sensitivity():
    tasks = []
    for rep in range(MC_REPS_SENS):
        for scenario in SCENARIOS_SENS:
            for rho in RHO_GRID_SENS:
                for nuisance_kind in NUISANCE_KINDS:
                    tasks.append((rep, scenario, rho, nuisance_kind))

    print(f"DR_CF sensitivity tasks: {len(tasks)}")
    print(f"MC_REPS_SENS={MC_REPS_SENS}, BOOT_REPS={BOOT_REPS}, N_JOBS={N_JOBS}")
    print(f"E_TRUNC_GRID={E_TRUNC_GRID}")
    print(f"NUISANCE_KINDS={NUISANCE_KINDS}")

    t0 = time.time()

    nested = Parallel(n_jobs=N_JOBS, verbose=10, backend="loky")(
        delayed(run_one_dr_cf_sensitivity_task)(rep, scenario, rho, nuisance_kind)
        for rep, scenario, rho, nuisance_kind in tasks
    )

    rows = [r for chunk in nested for r in chunk]
    df_sens = pd.DataFrame(rows)
    summary_sens = summarize_dr_cf_sensitivity(df_sens)

    detail_csv = os.path.join(OUTDIR_SENS, "dr_cf_sensitivity_detail.csv")
    summary_csv = os.path.join(OUTDIR_SENS, "dr_cf_sensitivity_summary.csv")

    df_sens.to_csv(detail_csv, index=False)
    summary_sens.to_csv(summary_csv, index=False)

    print("\nSaved:")
    print(detail_csv)
    print(summary_csv)
    print(f"Elapsed minutes: {(time.time() - t0) / 60.0:.2f}")

    print("\nSummary:")
    display(summary_sens.sort_values(["scenario", "rho", "nuisance", "e_trunc"]).round(4))

    return df_sens, summary_sens


# ------------------------------------------------------------
# SAFE 100/100 run block
# ------------------------------------------------------------
# This block is intentionally placed at the bottom of the same large cell,
# after all functions have been defined.  Do not move it to a separate cell
# unless the definition cell has already been executed.

import os
import time
from joblib import Parallel, delayed

# Mount Google Drive and use a fresh directory, so old 30-bootstrap checkpoints
# are not accidentally reused.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTDIR_SENS = "/content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2"
except Exception as e:
    print("Google Drive mount failed. Use local output instead.")
    print("Reason:", repr(e))
    OUTDIR_SENS = "zib_dr_cf_sensitivity_100_100_v2"

os.makedirs(OUTDIR_SENS, exist_ok=True)

DETAIL_CKPT = os.path.join(OUTDIR_SENS, "dr_cf_sensitivity_detail_checkpoint_100_100.csv")
DETAIL_FINAL = os.path.join(OUTDIR_SENS, "dr_cf_sensitivity_detail_100_100.csv")
SUMMARY_FINAL = os.path.join(OUTDIR_SENS, "dr_cf_sensitivity_summary_100_100.csv")

# Main bootstrap setting.
MC_REPS_SENS = 100
BOOT_REPS = 100
N_JOBS = 7

# Recommended main-text diagnostic.
# This checks the bootstrap CI behavior with the stable logistic nuisance.
# Runtime is roughly comparable to the previous pilot, but with true 100 bootstraps.
E_TRUNC_GRID = [0.001]
NUISANCE_KINDS = ["logistic"]

# For the full Appendix diagnostic, replace the two lines above by:
# E_TRUNC_GRID = [0.001, 0.01, 0.05]
# NUISANCE_KINDS = ["hgb", "logistic"]
# Be careful: the full version can be much longer.

CHUNK_REPS = 1
RESUME_FROM_CHECKPOINT = True

print("============================================================")
print("DR_CF sensitivity 100/100 run")
print("============================================================")
print("OUTDIR_SENS     =", OUTDIR_SENS)
print("MC_REPS_SENS    =", MC_REPS_SENS)
print("BOOT_REPS       =", BOOT_REPS)
print("N_JOBS          =", N_JOBS)
print("SCENARIOS_SENS  =", SCENARIOS_SENS)
print("RHO_GRID_SENS   =", RHO_GRID_SENS)
print("E_TRUNC_GRID    =", E_TRUNC_GRID)
print("NUISANCE_KINDS  =", NUISANCE_KINDS)
print("CHUNK_REPS      =", CHUNK_REPS)
print("RESUME          =", RESUME_FROM_CHECKPOINT)
print("============================================================")

# Build outer tasks.  Each outer task returns rows over E_TRUNC_GRID.
all_tasks = []
for rep in range(MC_REPS_SENS):
    for scenario in SCENARIOS_SENS:
        for rho in RHO_GRID_SENS:
            for nuisance_kind in NUISANCE_KINDS:
                all_tasks.append((rep, scenario, rho, nuisance_kind))

print("Total outer tasks =", len(all_tasks))
print("Approx bootstrap fits =", len(all_tasks) * len(E_TRUNC_GRID) * BOOT_REPS)

# Resume from checkpoint, if this exact v2 directory already has partial output.
if RESUME_FROM_CHECKPOINT and os.path.exists(DETAIL_CKPT):
    df_done = pd.read_csv(DETAIL_CKPT)
    print("Loaded checkpoint:", DETAIL_CKPT)
    print("Existing rows:", len(df_done))

    done_keys = set(
        zip(
            df_done["rep"].astype(int),
            df_done["scenario"].astype(str),
            df_done["rho"].astype(float),
            df_done["nuisance"].astype(str),
        )
    )

    tasks = [
        t for t in all_tasks
        if (int(t[0]), str(t[1]), float(t[2]), str(t[3])) not in done_keys
    ]
else:
    df_done = pd.DataFrame()
    tasks = all_tasks

print("Remaining outer tasks =", len(tasks))

t0 = time.time()

if len(tasks) == 0:
    print("No remaining tasks. Using checkpoint.")
    df_sens = df_done.copy()
else:
    rows_accum = [] if df_done.empty else [df_done]
    reps_remaining = sorted(set(int(t[0]) for t in tasks))

    for chunk_start in range(0, len(reps_remaining), CHUNK_REPS):
        rep_chunk = reps_remaining[chunk_start:chunk_start + CHUNK_REPS]
        chunk_tasks = [t for t in tasks if int(t[0]) in rep_chunk]

        print("------------------------------------------------------------")
        print("Running MC reps:", rep_chunk)
        print("Chunk outer tasks:", len(chunk_tasks))
        print("Elapsed so far: %.2f min" % ((time.time() - t0) / 60.0))
        print("------------------------------------------------------------")

        nested = Parallel(n_jobs=N_JOBS, verbose=10, backend="loky")(
            delayed(run_one_dr_cf_sensitivity_task)(rep, scenario, rho, nuisance_kind)
            for rep, scenario, rho, nuisance_kind in chunk_tasks
        )

        chunk_rows = [r for chunk in nested for r in chunk]
        df_chunk = pd.DataFrame(chunk_rows)

        rows_accum.append(df_chunk)
        df_sens = pd.concat(rows_accum, ignore_index=True)

        # Save checkpoint after each MC replication.
        df_sens.to_csv(DETAIL_CKPT, index=False)
        print("Checkpoint saved:", DETAIL_CKPT)
        print("Rows saved:", len(df_sens))

        # Immediate bootstrap-count check.
        if "boot_success" in df_chunk.columns:
            bs = df_chunk["boot_success"].dropna()
            if len(bs) > 0:
                print(
                    "Bootstrap success in this chunk: "
                    f"min={bs.min():.0f}, mean={bs.mean():.2f}, max={bs.max():.0f}"
                )
                if BOOT_REPS == 100 and bs.max() <= 30:
                    raise RuntimeError(
                        "BOOT_REPS is set to 100, but boot_success never exceeds 30. "
                        "This means an old 30-bootstrap setting is still being used. "
                        "Search for 'BOOT_REPS = 30' or use the fixed notebook."
                    )

summary_sens = summarize_dr_cf_sensitivity(df_sens)

df_sens.to_csv(DETAIL_FINAL, index=False)
summary_sens.to_csv(SUMMARY_FINAL, index=False)

print("============================================================")
print("Finished DR_CF sensitivity run")
print("============================================================")
print("Elapsed total: %.2f min" % ((time.time() - t0) / 60.0))
print("Detail saved to :", DETAIL_FINAL)
print("Summary saved to:", SUMMARY_FINAL)
print("Detail shape    :", df_sens.shape)
print("Summary shape   :", summary_sens.shape)

print("------------------------------------------------------------")
print("Bootstrap success summary")
print("------------------------------------------------------------")
print(df_sens["boot_success"].describe())

if BOOT_REPS == 100 and df_sens["boot_success"].max() <= 30:
    raise RuntimeError(
        "Final boot_success is still capped around 30 although BOOT_REPS=100. "
        "Please use the fixed notebook or remove the old BOOT_REPS=30 setting."
    )

if BOOT_REPS == 100 and summary_sens["boot_success_mean"].mean() < 80:
    print(
        "\nWarning: average bootstrap success is below 80. "
        "This may be due to failed bootstrap fits, especially in low-rho cases."
    )

cols_show = [
    "scenario", "rho", "nuisance", "e_trunc", "n_rep",
    "rmse_mean", "bias_mean",
    "cover_sandwich", "cover_boot_pct",
    "width_sandwich", "width_boot_pct",
    "boot_success_mean",
    "ess_mean", "n_revealed_mean",
    "ehat_q01_mean", "ehat_q05_mean",
]

display(summary_sens[cols_show])


Mounted at /content/drive
DR_CF sensitivity 100/100 run
OUTDIR_SENS     = /content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2
MC_REPS_SENS    = 100
BOOT_REPS       = 100
N_JOBS          = 7
SCENARIOS_SENS  = ['clean', 'informative_H']
RHO_GRID_SENS   = [0.01, 0.02, 0.05, 0.1, 0.2]
E_TRUNC_GRID    = [0.001]
NUISANCE_KINDS  = ['logistic']
CHUNK_REPS      = 1
RESUME          = True
Total outer tasks = 1000
Approx bootstrap fits = 100000
Remaining outer tasks = 1000
------------------------------------------------------------
Running MC reps: [0]
Chunk outer tasks: 10
Elapsed so far: 0.00 min
------------------------------------------------------------


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   3 out of  10 | elapsed:   13.8s remaining:   32.3s
[Parallel(n_jobs=7)]: Done   5 out of  10 | elapsed:   14.6s remaining:   14.6s
[Parallel(n_jobs=7)]: Done   7 out of  10 | elapsed:   16.5s remaining:    7.1s
[Parallel(n_jobs=7)]: Done  10 out of  10 | elapsed:   22.8s finished
[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


Checkpoint saved: /content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_detail_checkpoint_100_100.csv
Rows saved: 10
Bootstrap success in this chunk: min=100, mean=100.00, max=100
------------------------------------------------------------
Running MC reps: [1]
Chunk outer tasks: 10
Elapsed so far: 0.38 min
------------------------------------------------------------


[Parallel(n_jobs=7)]: Done   3 out of  10 | elapsed:   11.8s remaining:   27.6s
[Parallel(n_jobs=7)]: Done   5 out of  10 | elapsed:   12.7s remaining:   12.7s
[Parallel(n_jobs=7)]: Done   7 out of  10 | elapsed:   16.8s remaining:    7.2s
[Parallel(n_jobs=7)]: Done  10 out of  10 | elapsed:   20.3s finished
[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


Checkpoint saved: /content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_detail_checkpoint_100_100.csv
Rows saved: 20
Bootstrap success in this chunk: min=100, mean=100.00, max=100
------------------------------------------------------------
Running MC reps: [2]
Chunk outer tasks: 10
Elapsed so far: 0.72 min
------------------------------------------------------------


...

[Parallel(n_jobs=7)]: Done   3 out of  10 | elapsed:   12.4s remaining:   29.0s
[Parallel(n_jobs=7)]: Done   5 out of  10 | elapsed:   13.0s remaining:   13.0s
[Parallel(n_jobs=7)]: Done   7 out of  10 | elapsed:   13.7s remaining:    5.9s
[Parallel(n_jobs=7)]: Done  10 out of  10 | elapsed:   20.4s finished
[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


Checkpoint saved: /content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_detail_checkpoint_100_100.csv
Rows saved: 990
Bootstrap success in this chunk: min=100, mean=100.00, max=100
------------------------------------------------------------
Running MC reps: [99]
Chunk outer tasks: 10
Elapsed so far: 33.96 min
------------------------------------------------------------


[Parallel(n_jobs=7)]: Done   3 out of  10 | elapsed:   11.4s remaining:   26.5s
[Parallel(n_jobs=7)]: Done   5 out of  10 | elapsed:   12.9s remaining:   12.9s
[Parallel(n_jobs=7)]: Done   7 out of  10 | elapsed:   16.3s remaining:    7.0s
[Parallel(n_jobs=7)]: Done  10 out of  10 | elapsed:   20.3s finished


Checkpoint saved: /content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_detail_checkpoint_100_100.csv
Rows saved: 1000
Bootstrap success in this chunk: min=100, mean=100.00, max=100
Finished DR_CF sensitivity run
Elapsed total: 34.30 min
Detail saved to : /content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_detail_100_100.csv
Summary saved to: /content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_summary_100_100.csv
Detail shape    : (1000, 28)
Summary shape   : (10, 18)
------------------------------------------------------------
Bootstrap success summary
------------------------------------------------------------
count    1000.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
Name: boot_success, dtype: float64


,scenario,rho,nuisance,e_trunc,n_rep,rmse_mean,bias_mean,cover_sandwich,cover_boot_pct,width_sandwich,width_boot_pct,boot_success_mean,ess_mean,n_revealed_mean,ehat_q01_mean,ehat_q05_mean
0,clean,0.01,logistic,0.001,100,0.347470,0.297274,0.24250,0.91250,0.207324,1.320518,100.0,9.099608,20.71,0.001010,0.001138
1,clean,0.02,logistic,0.001,100,0.308347,0.262150,0.25500,0.91500,0.206385,1.101644,100.0,14.148931,40.66,0.001086,0.001608
2,clean,0.05,logistic,0.001,100,0.204180,0.172072,0.36875,0.92250,0.214662,0.774438,100.0,34.610812,103.14,0.001887,0.004103
3,clean,0.10,logistic,0.001,100,0.156264,0.131309,0.48750,0.91750,0.221475,0.580602,100.0,71.266647,205.85,0.004193,0.009297
4,clean,0.20,logistic,0.001,100,0.108652,0.089261,0.69750,0.93250,0.232075,0.433149,100.0,170.639589,412.85,0.011271,0.024625
5,informative_H,0.01,logistic,0.001,100,1.529151,1.009380,0.07000,0.43375,0.179758,1.577033,100.0,3.827493,20.02,0.001000,0.001000
6,informative_H,0.02,logistic,0.001,100,0.947879,0.674146,0.13750,0.59125,0.196370,1.750768,100.0,5.832758,41.04,0.001000,0.001000
7,informative_H,0.05,logistic,0.001,100,0.303470,0.252547,0.34875,0.83875,0.232644,0.976085,100.0,7.885087,102.91,0.001000,0.001000
8,informative_H,0.10,logistic,0.001,100,0.144706,0.120774,0.58875,0.93625,0.251175,0.554390,100.0,13.724540,206.65,0.001000,0.001000
9,informative_H,0.20,logistic,0.001,100,0.096479,0.079071,0.81125,0.92500,0.262472,0.375172,100.0,31.992065,411.36,0.001000,0.001069


In [ ]:
# ============================================================
# Additional bootstrap coverage for STD_ZIB and PR_MLE
# This cell assumes that the previous DR_CF runtime is still alive.
# It does NOT rerun DR_CF.
# ============================================================

import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

# Avoid thread oversubscription
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ------------------------------------------------------------
# 0. Output directory
# ------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTDIR_LIKBOOT = "/content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2"
except Exception as e:
    print("Google Drive mount failed. Use local output instead.")
    print("Reason:", repr(e))
    OUTDIR_LIKBOOT = "zib_likelihood_bootstrap_100_100_v2"

os.makedirs(OUTDIR_LIKBOOT, exist_ok=True)

DETAIL_CKPT_LIK = os.path.join(OUTDIR_LIKBOOT, "likelihood_boot_detail_checkpoint_100_100.csv")
DETAIL_FINAL_LIK = os.path.join(OUTDIR_LIKBOOT, "likelihood_boot_detail_100_100.csv")
SUMMARY_FINAL_LIK = os.path.join(OUTDIR_LIKBOOT, "likelihood_boot_summary_100_100.csv")
TABLE1_COMPLETED = os.path.join(OUTDIR_LIKBOOT, "table1_with_likelihood_and_dr_bootstrap_100_100.csv")

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
MC_REPS_LIKBOOT = 100
BOOT_REPS_LIKBOOT = 100
N_JOBS_LIKBOOT = 7

SCENARIOS_LIKBOOT = ["clean", "informative_H"]
RHO_GRID_LIKBOOT = [0.01, 0.02, 0.05, 0.10, 0.20]

# Same seed convention as DR_CF sensitivity run
BASE_SEED_LIKBOOT = 2026

# Save every 7 MC replications
CHUNK_REPS_LIKBOOT = 7
RESUME_FROM_CHECKPOINT_LIKBOOT = True

# Constants if not already present
if "N_SENS" not in globals():
    N_SENS = globals().get("N", 2500)

if "MAX_ABS_THETA" not in globals():
    MAX_ABS_THETA = 30.0

if "ALPHA_TRUE" not in globals():
    ALPHA_TRUE = np.array([-0.8,  1.2, -0.9,  0.6], dtype=float)

if "BETA_TRUE" not in globals():
    BETA_TRUE  = np.array([ 0.2,  0.5,  1.0, -0.6], dtype=float)

# ------------------------------------------------------------
# 2. Required definitions check
# ------------------------------------------------------------
required_names = [
    "simulate_structural_zib",
    "add_auxiliary_history",
    "attach_partial_revelation",
    "fit_standard_once",
    "fit_pr_mle",
    "standard_loss_grad",
    "pr_loss_grad",
    "multi_start_fit",
    "fit_bfgs",
    "pinv_psd",
    "numerical_hessian_from_grad",
]

missing = [name for name in required_names if name not in globals()]
if len(missing) > 0:
    raise RuntimeError(
        "Missing required definitions: "
        + ", ".join(missing)
        + "\nRun the main self-contained simulation cell first. "
          "Do not restart the runtime before this likelihood-bootstrap cell."
    )

print("============================================================")
print("Likelihood bootstrap for STD_ZIB and PR_MLE")
print("============================================================")
print("OUTDIR_LIKBOOT      =", OUTDIR_LIKBOOT)
print("MC_REPS_LIKBOOT     =", MC_REPS_LIKBOOT)
print("BOOT_REPS_LIKBOOT   =", BOOT_REPS_LIKBOOT)
print("N_JOBS_LIKBOOT      =", N_JOBS_LIKBOOT)
print("N_SENS              =", N_SENS)
print("SCENARIOS_LIKBOOT   =", SCENARIOS_LIKBOOT)
print("RHO_GRID_LIKBOOT    =", RHO_GRID_LIKBOOT)
print("CHUNK_REPS_LIKBOOT  =", CHUNK_REPS_LIKBOOT)
print("============================================================")

# ------------------------------------------------------------
# 3. Utilities
# ------------------------------------------------------------
def _subset_data_local(data, idx):
    """
    Subset arrays whose first dimension equals n.
    Keep scalars and fixed-length parameter vectors unchanged.
    """
    n = len(data["Y"])
    out = {}
    for k, v in data.items():
        if isinstance(v, np.ndarray) and v.shape[0] == n:
            out[k] = v[idx]
        else:
            out[k] = v
    return out


def _safe_coverage(theta_true, lo, hi):
    theta_true = np.asarray(theta_true)
    lo = np.asarray(lo)
    hi = np.asarray(hi)
    ok = np.isfinite(lo) & np.isfinite(hi)
    if ok.sum() == 0:
        return np.nan
    return float(((theta_true[ok] >= lo[ok]) & (theta_true[ok] <= hi[ok])).mean())


def _ci_width_local(lo, hi):
    lo = np.asarray(lo)
    hi = np.asarray(hi)
    ok = np.isfinite(lo) & np.isfinite(hi)
    if ok.sum() == 0:
        return np.nan
    return float(np.mean(hi[ok] - lo[ok]))


def fit_standard_theta_only(base, theta_start, start_seed=123):
    """
    Faster bootstrap refit for STD_ZIB: returns theta only, no Hessian.
    """
    X = base["X"]
    Y = base["Y"]
    p = X.shape[1]

    starts = [
        np.asarray(theta_start, dtype=float),
        np.zeros(2 * p),
        np.r_[theta_start[p:], theta_start[:p]],
    ]

    # add a small random start for robustness
    rng = np.random.default_rng(start_seed)
    starts.append(rng.normal(scale=0.5, size=2 * p))

    res = multi_start_fit(lambda th: standard_loss_grad(th, X, Y), starts)
    theta = res.x

    if (not np.all(np.isfinite(theta))) or np.max(np.abs(theta)) > MAX_ABS_THETA:
        raise RuntimeError("STD_ZIB bootstrap fit is non-finite or exploding.")

    return theta


def fit_pr_theta_only(data, theta_start, start_seed=123):
    """
    Faster bootstrap refit for PR_MLE: returns theta only, no Hessian.
    """
    X = data["X"]
    Y = data["Y"]
    R = data["R"]
    W = data["W"]
    p = X.shape[1]

    starts = [
        np.asarray(theta_start, dtype=float),
        np.zeros(2 * p),
        np.r_[theta_start[p:], theta_start[:p]],
    ]

    rng = np.random.default_rng(start_seed)
    starts.append(theta_start + rng.normal(scale=0.2, size=2 * p))

    res = multi_start_fit(lambda th: pr_loss_grad(th, X, Y, R, W), starts)
    theta = res.x

    if (not np.all(np.isfinite(theta))) or np.max(np.abs(theta)) > MAX_ABS_THETA:
        raise RuntimeError("PR_MLE bootstrap fit is non-finite or exploding.")

    return theta


def bootstrap_percentile_ci_likelihood(data, method, theta_hat, B, seed):
    """
    Nonparametric pairs bootstrap percentile CI for STD_ZIB or PR_MLE.
    Nuisance functions are not involved here.
    """
    rng = np.random.default_rng(seed)
    n = len(data["Y"])

    boot_thetas = []
    n_fail = 0

    for b in range(B):
        idx = rng.integers(0, n, size=n)
        db = _subset_data_local(data, idx)

        try:
            if method == "STD_ZIB":
                thb = fit_standard_theta_only(
                    db,
                    theta_start=theta_hat,
                    start_seed=seed + 1009 * b + 11
                )
            elif method == "PR_MLE":
                thb = fit_pr_theta_only(
                    db,
                    theta_start=theta_hat,
                    start_seed=seed + 1009 * b + 17
                )
            else:
                raise ValueError("Unknown method: " + str(method))

            boot_thetas.append(thb)

        except Exception:
            n_fail += 1

    if len(boot_thetas) == 0:
        d = len(theta_hat)
        return {
            "lo": np.full(d, np.nan),
            "hi": np.full(d, np.nan),
            "n_success": 0,
            "n_fail": int(n_fail),
        }

    boot_thetas = np.vstack(boot_thetas)
    lo = np.nanpercentile(boot_thetas, 2.5, axis=0)
    hi = np.nanpercentile(boot_thetas, 97.5, axis=0)

    return {
        "lo": lo,
        "hi": hi,
        "n_success": int(len(boot_thetas)),
        "n_fail": int(n_fail),
    }


def one_likelihood_bootstrap_rep(rep):
    """
    One MC replication.
    Returns:
      - STD_ZIB rows, duplicated for clean and informative_H at rho=0
      - PR_MLE rows for each scenario and rho
    """
    theta_true = np.r_[ALPHA_TRUE, BETA_TRUE]
    p = len(ALPHA_TRUE)

    rows = []

    # Same structural data as DR_CF sensitivity
    structural = simulate_structural_zib(
        n=N_SENS,
        alpha=ALPHA_TRUE,
        beta=BETA_TRUE,
        seed=BASE_SEED_LIKBOOT + 1000 * rep
    )

    # --------------------------------------------------------
    # STD_ZIB: fit once, bootstrap once, duplicate scenario label
    # --------------------------------------------------------
    try:
        std_fit = fit_standard_once(
            structural,
            start_seed=BASE_SEED_LIKBOOT + rep
        )

        theta_hat = std_fit["theta"]
        se = np.clip(std_fit["se"], 1e-8, np.inf)

        sand_lo = theta_hat - 1.96 * se
        sand_hi = theta_hat + 1.96 * se

        boot = bootstrap_percentile_ci_likelihood(
            structural,
            method="STD_ZIB",
            theta_hat=theta_hat,
            B=BOOT_REPS_LIKBOOT,
            seed=BASE_SEED_LIKBOOT + 700000 + 1000 * rep + 31
        )

        boot_lo = boot["lo"]
        boot_hi = boot["hi"]

        err = theta_hat - theta_true
        d_swap = np.r_[np.ones(p), -np.ones(p)] / np.sqrt(2 * p)
        min_eig = float(np.linalg.eigvalsh(0.5 * (std_fit["H"] + std_fit["H"].T)).min())

        for scenario in SCENARIOS_LIKBOOT:
            rows.append({
                "scenario": scenario,
                "rho": 0.00,
                "rep": int(rep),
                "method": "STD_ZIB",
                "rmse_all": float(np.sqrt(np.mean(err ** 2))),
                "mean_abs_bias": float(np.mean(np.abs(err))),
                "coverage_sandwich": _safe_coverage(theta_true, sand_lo, sand_hi),
                "coverage_boot_pct": _safe_coverage(theta_true, boot_lo, boot_hi),
                "width_sandwich": _ci_width_local(sand_lo, sand_hi),
                "width_boot_pct": _ci_width_local(boot_lo, boot_hi),
                "boot_success": int(boot["n_success"]),
                "boot_fail": int(boot["n_fail"]),
                "min_eig": min_eig,
                "swap_proj": float(d_swap @ theta_hat),
                "alpha1_hat": float(theta_hat[1]),
                "beta1_hat": float(theta_hat[p + 1]),
                "error": "",
            })

    except Exception as exc:
        for scenario in SCENARIOS_LIKBOOT:
            rows.append({
                "scenario": scenario,
                "rho": 0.00,
                "rep": int(rep),
                "method": "STD_ZIB",
                "error": str(exc)[:300],
            })

        # PR_MLE needs a starting value; if STD failed, use zero start.
        std_fit = {"theta": np.zeros(2 * p)}

    # --------------------------------------------------------
    # PR_MLE: scenario/rho-specific partial revelation
    # --------------------------------------------------------
    for scenario in SCENARIOS_LIKBOOT:
        base = add_auxiliary_history(
            structural,
            scenario=scenario,
            seed=BASE_SEED_LIKBOOT + 1000 * rep + 10000 * SCENARIOS_LIKBOOT.index(scenario) + 17
        )

        for rho in RHO_GRID_LIKBOOT:
            try:
                data = attach_partial_revelation(
                    base,
                    rho_target=rho,
                    seed=BASE_SEED_LIKBOOT
                         + 1000 * rep
                         + 10000 * SCENARIOS_LIKBOOT.index(scenario)
                         + int(1e5 * rho)
                         + 13
                )

                pr_fit = fit_pr_mle(data, theta_init=std_fit["theta"])

                theta_hat = pr_fit["theta"]
                se = np.clip(pr_fit["se"], 1e-8, np.inf)

                sand_lo = theta_hat - 1.96 * se
                sand_hi = theta_hat + 1.96 * se

                boot = bootstrap_percentile_ci_likelihood(
                    data,
                    method="PR_MLE",
                    theta_hat=theta_hat,
                    B=BOOT_REPS_LIKBOOT,
                    seed=BASE_SEED_LIKBOOT
                         + 800000
                         + 1000 * rep
                         + 10000 * SCENARIOS_LIKBOOT.index(scenario)
                         + int(1e5 * rho)
                         + 47
                )

                boot_lo = boot["lo"]
                boot_hi = boot["hi"]

                err = theta_hat - theta_true
                d_swap = np.r_[np.ones(p), -np.ones(p)] / np.sqrt(2 * p)
                min_eig = float(np.linalg.eigvalsh(0.5 * (pr_fit["H"] + pr_fit["H"].T)).min())

                rows.append({
                    "scenario": scenario,
                    "rho": float(rho),
                    "rep": int(rep),
                    "method": "PR_MLE",
                    "rho_realized": float(data.get("rho_realized", np.nan)),
                    "rmse_all": float(np.sqrt(np.mean(err ** 2))),
                    "mean_abs_bias": float(np.mean(np.abs(err))),
                    "coverage_sandwich": _safe_coverage(theta_true, sand_lo, sand_hi),
                    "coverage_boot_pct": _safe_coverage(theta_true, boot_lo, boot_hi),
                    "width_sandwich": _ci_width_local(sand_lo, sand_hi),
                    "width_boot_pct": _ci_width_local(boot_lo, boot_hi),
                    "boot_success": int(boot["n_success"]),
                    "boot_fail": int(boot["n_fail"]),
                    "min_eig": min_eig,
                    "swap_proj": float(d_swap @ theta_hat),
                    "alpha1_hat": float(theta_hat[1]),
                    "beta1_hat": float(theta_hat[p + 1]),
                    "n_revealed": int(np.sum(data["R"])),
                    "error": "",
                })

            except Exception as exc:
                rows.append({
                    "scenario": scenario,
                    "rho": float(rho),
                    "rep": int(rep),
                    "method": "PR_MLE",
                    "error": str(exc)[:300],
                })

    return rows


def summarize_likelihood_bootstrap(df):
    ok = df.copy()

    # Drop rows with fitting errors for numerical summaries
    if "error" in ok.columns:
        ok = ok[(ok["error"].isna()) | (ok["error"].astype(str) == "")].copy()

    return (
        ok.groupby(["scenario", "rho", "method"], as_index=False)
          .agg(
              n_rep=("rep", "nunique"),
              rmse_mean=("rmse_all", "mean"),
              rmse_sd=("rmse_all", "std"),
              bias_mean=("mean_abs_bias", "mean"),
              cover_sandwich=("coverage_sandwich", "mean"),
              cover_boot_pct=("coverage_boot_pct", "mean"),
              width_sandwich=("width_sandwich", "mean"),
              width_boot_pct=("width_boot_pct", "mean"),
              boot_success_mean=("boot_success", "mean"),
              boot_fail_mean=("boot_fail", "mean"),
              min_eig_mean=("min_eig", "mean"),
          )
    )


# ------------------------------------------------------------
# 4. Resume checkpoint
# ------------------------------------------------------------
all_reps = list(range(MC_REPS_LIKBOOT))

if RESUME_FROM_CHECKPOINT_LIKBOOT and os.path.exists(DETAIL_CKPT_LIK):
    df_done = pd.read_csv(DETAIL_CKPT_LIK)
    done_reps = sorted(df_done["rep"].dropna().astype(int).unique().tolist())
    reps_remaining = [r for r in all_reps if r not in set(done_reps)]
    print("Loaded checkpoint:", DETAIL_CKPT_LIK)
    print("Existing rows:", len(df_done))
    print("Completed reps:", len(done_reps))
else:
    df_done = pd.DataFrame()
    reps_remaining = all_reps

print("Remaining reps:", len(reps_remaining))

# ------------------------------------------------------------
# 5. Run in chunks
# ------------------------------------------------------------
t0 = time.time()

if len(reps_remaining) == 0:
    df_likboot = df_done.copy()
else:
    chunks_accum = [] if df_done.empty else [df_done]

    for start in range(0, len(reps_remaining), CHUNK_REPS_LIKBOOT):
        rep_chunk = reps_remaining[start:start + CHUNK_REPS_LIKBOOT]

        print("------------------------------------------------------------")
        print("Running likelihood bootstrap MC reps:", rep_chunk)
        print("Elapsed so far: %.2f min" % ((time.time() - t0) / 60.0))
        print("------------------------------------------------------------")

        nested = Parallel(n_jobs=N_JOBS_LIKBOOT, verbose=5, backend="loky")(
            delayed(one_likelihood_bootstrap_rep)(rep)
            for rep in rep_chunk
        )

        rows_chunk = [r for rep_rows in nested for r in rep_rows]
        df_chunk = pd.DataFrame(rows_chunk)

        chunks_accum.append(df_chunk)
        df_likboot = pd.concat(chunks_accum, ignore_index=True)

        df_likboot.to_csv(DETAIL_CKPT_LIK, index=False)

        print("Checkpoint saved:", DETAIL_CKPT_LIK)
        print("Rows saved:", len(df_likboot))

        if "boot_success" in df_chunk.columns:
            bs = pd.to_numeric(df_chunk["boot_success"], errors="coerce").dropna()
            if len(bs) > 0:
                print(
                    "Bootstrap success in this chunk: "
                    f"min={bs.min():.0f}, mean={bs.mean():.2f}, max={bs.max():.0f}"
                )

# ------------------------------------------------------------
# 6. Summarize likelihood bootstrap
# ------------------------------------------------------------
summary_likboot = summarize_likelihood_bootstrap(df_likboot)

df_likboot.to_csv(DETAIL_FINAL_LIK, index=False)
summary_likboot.to_csv(SUMMARY_FINAL_LIK, index=False)

print("============================================================")
print("Finished likelihood bootstrap")
print("============================================================")
print("Elapsed total: %.2f min" % ((time.time() - t0) / 60.0))
print("Detail saved to :", DETAIL_FINAL_LIK)
print("Summary saved to:", SUMMARY_FINAL_LIK)
print("Detail shape    :", df_likboot.shape)
print("Summary shape   :", summary_likboot.shape)

print("------------------------------------------------------------")
print("Likelihood bootstrap success summary")
print("------------------------------------------------------------")
print(df_likboot.groupby("method")["boot_success"].describe())

# ------------------------------------------------------------
# 7. Combine with already computed DR_CF summary if available
# ------------------------------------------------------------
def _get_dr_summary_from_runtime_or_file():
    # Prefer current runtime object
    if "summary_sens" in globals():
        dr = summary_sens.copy()
        print("Using DR_CF summary from current runtime: summary_sens")
        return dr

    # Otherwise try common saved paths
    candidate_paths = [
        "/content/drive/MyDrive/zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_summary_100_100.csv",
        "/content/drive/MyDrive/zib_dr_cf_sensitivity_100_100/dr_cf_sensitivity_summary_100_100.csv",
        "zib_dr_cf_sensitivity_100_100_v2/dr_cf_sensitivity_summary_100_100.csv",
    ]

    for path in candidate_paths:
        if os.path.exists(path):
            print("Using DR_CF summary from file:", path)
            return pd.read_csv(path)

    print("DR_CF summary not found. Only likelihood bootstrap summary will be saved.")
    return None


dr_summary = _get_dr_summary_from_runtime_or_file()

if dr_summary is not None:
    dr = dr_summary.copy()

    # Keep the main-table DR_CF setting
    if "nuisance" in dr.columns:
        dr = dr[dr["nuisance"].astype(str) == "logistic"].copy()
    if "e_trunc" in dr.columns:
        dr = dr[np.isclose(dr["e_trunc"].astype(float), 0.001)].copy()

    dr_table = pd.DataFrame({
        "scenario": dr["scenario"],
        "rho": dr["rho"].astype(float),
        "method": "DR_CF",
        "n_rep": dr["n_rep"],
        "rmse_mean": dr["rmse_mean"],
        "rmse_sd": dr.get("rmse_sd", np.nan),
        "bias_mean": dr.get("bias_mean", np.nan),
        "cover_sandwich": dr["cover_sandwich"],
        "cover_boot_pct": dr["cover_boot_pct"],
        "width_sandwich": dr.get("width_sandwich", np.nan),
        "width_boot_pct": dr.get("width_boot_pct", np.nan),
        "boot_success_mean": dr.get("boot_success_mean", np.nan),
        "boot_fail_mean": np.nan,
        "min_eig_mean": np.nan,
    })

    lik_table = summary_likboot.copy()

    table1_completed = pd.concat([lik_table, dr_table], ignore_index=True)

else:
    table1_completed = summary_likboot.copy()

# Nice order
method_order = {"STD_ZIB": 0, "PR_MLE": 1, "DR_CF": 2}
scenario_order = {"clean": 0, "informative_H": 1}

table1_completed["_scenario_order"] = table1_completed["scenario"].map(scenario_order)
table1_completed["_method_order"] = table1_completed["method"].map(method_order)
table1_completed = table1_completed.sort_values(
    ["_scenario_order", "_method_order", "rho"]
).drop(columns=["_scenario_order", "_method_order"])

table1_completed.to_csv(TABLE1_COMPLETED, index=False)

print("------------------------------------------------------------")
print("Completed Table 1 summary saved to:")
print(TABLE1_COMPLETED)
print("------------------------------------------------------------")

cols_show = [
    "scenario", "rho", "method", "n_rep",
    "rmse_mean", "bias_mean",
    "cover_sandwich", "cover_boot_pct",
    "width_sandwich", "width_boot_pct",
    "boot_success_mean", "min_eig_mean",
]

display(table1_completed[cols_show])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Likelihood bootstrap for STD_ZIB and PR_MLE
OUTDIR_LIKBOOT      = /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2
MC_REPS_LIKBOOT     = 100
BOOT_REPS_LIKBOOT   = 100
N_JOBS_LIKBOOT      = 7
N_SENS              = 2500
SCENARIOS_LIKBOOT   = ['clean', 'informative_H']
RHO_GRID_LIKBOOT    = [0.01, 0.02, 0.05, 0.1, 0.2]
CHUNK_REPS_LIKBOOT  = 7
Remaining reps: 100
------------------------------------------------------------
Running likelihood bootstrap MC reps: [0, 1, 2, 3, 4, 5, 6]
Elapsed so far: 0.00 min
------------------------------------------------------------


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   2 out of   7 | elapsed:  5.4min remaining: 13.5min
[Parallel(n_jobs=7)]: Done   4 out of   7 | elapsed:  5.4min remaining:  4.1min
[Parallel(n_jobs=7)]: Done   7 out of   7 | elapsed:  5.5min finished
[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


Checkpoint saved: /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2/likelihood_boot_detail_checkpoint_100_100.csv
Rows saved: 84
Bootstrap success in this chunk: min=98, mean=99.89, max=100
------------------------------------------------------------
Running likelihood bootstrap MC reps: [7, 8, 9, 10, 11, 12, 13]
Elapsed so far: 5.52 min
------------------------------------------------------------


[Parallel(n_jobs=7)]: Done   2 out of   7 | elapsed:  5.2min remaining: 13.1min
[Parallel(n_jobs=7)]: Done   4 out of   7 | elapsed:  5.4min remaining:  4.0min
[Parallel(n_jobs=7)]: Done   7 out of   7 | elapsed:  5.5min finished
[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


Checkpoint saved: /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2/likelihood_boot_detail_checkpoint_100_100.csv
Rows saved: 168
Bootstrap success in this chunk: min=96, mean=99.93, max=100
------------------------------------------------------------
Running likelihood bootstrap MC reps: [14, 15, 16, 17, 18, 19, 20]
Elapsed so far: 11.06 min
------------------------------------------------------------


...

Checkpoint saved: /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2/likelihood_boot_detail_checkpoint_100_100.csv
Rows saved: 1092
Bootstrap success in this chunk: min=99, mean=99.93, max=100
------------------------------------------------------------
Running likelihood bootstrap MC reps: [91, 92, 93, 94, 95, 96, 97]
Elapsed so far: 71.52 min
------------------------------------------------------------


[Parallel(n_jobs=7)]: Done   2 out of   7 | elapsed:  5.2min remaining: 13.1min
[Parallel(n_jobs=7)]: Done   4 out of   7 | elapsed:  5.4min remaining:  4.0min
[Parallel(n_jobs=7)]: Done   7 out of   7 | elapsed:  5.5min finished
[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


Checkpoint saved: /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2/likelihood_boot_detail_checkpoint_100_100.csv
Rows saved: 1176
Bootstrap success in this chunk: min=100, mean=100.00, max=100
------------------------------------------------------------
Running likelihood bootstrap MC reps: [98, 99]
Elapsed so far: 77.00 min
------------------------------------------------------------
Checkpoint saved: /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2/likelihood_boot_detail_checkpoint_100_100.csv
Rows saved: 1200
Bootstrap success in this chunk: min=97, mean=99.75, max=100
Finished likelihood bootstrap
Elapsed total: 80.32 min
Detail saved to : /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2/likelihood_boot_detail_100_100.csv
Summary saved to: /content/drive/MyDrive/zib_likelihood_bootstrap_100_100_v2/likelihood_boot_summary_100_100.csv
Detail shape    : (1200, 19)
Summary shape   : (12, 14)
------------------------------------------------------------
Like

[Parallel(n_jobs=7)]: Done   2 out of   2 | elapsed:  3.3min finished


,scenario,rho,method,n_rep,rmse_mean,bias_mean,cover_sandwich,cover_boot_pct,width_sandwich,width_boot_pct,boot_success_mean,min_eig_mean
0,clean,0.00,STD_ZIB,100,1.024918,0.877844,0.49375,0.93125,1.449631,3.306461,99.62,0.873727
1,clean,0.01,PR_MLE,100,0.225230,0.181755,0.95625,0.94750,0.895395,1.091040,100.00,2.556637
2,clean,0.02,PR_MLE,100,0.176004,0.143988,0.96125,0.93750,0.740128,0.785495,100.00,4.319505
3,clean,0.05,PR_MLE,100,0.137815,0.113170,0.96750,0.94250,0.591577,0.594149,100.00,8.387986
4,clean,0.10,PR_MLE,100,0.122018,0.100681,0.96000,0.94500,0.501604,0.493584,100.00,13.199217
5,clean,0.20,PR_MLE,100,0.099470,0.080599,0.95125,0.93500,0.422463,0.414171,100.00,20.851104
12,clean,0.01,DR_CF,100,0.347470,0.297274,0.24250,0.91250,0.207324,1.320518,100.00,NaN
13,clean,0.02,DR_CF,100,0.308347,0.262150,0.25500,0.91500,0.206385,1.101644,100.00,NaN
14,clean,0.05,DR_CF,100,0.204180,0.172072,0.36875,0.92250,0.214662,0.774438,100.00,NaN
15,clean,0.10,DR_CF,100,0.156264,0.131309,0.48750,0.91750,0.221475,0.580602,100.00,NaN
